### 1. Load Aspect-Tagged Data

In [1]:
import pandas as pd
import os

# ==========================================
# 1. LOAD THE ASPECT-EXTRACTED DATA
# ==========================================
input_csv = "../data/comment_and_aspects/aspect_detection_results.csv"
print(f"🔄 Loading data from: {input_csv}")

try:
    df = pd.read_csv(input_csv)
    print(f"✅ Successfully loaded {len(df)} aspect-tagged comments.")
    print(f"📋 Available columns: {list(df.columns)}")
except FileNotFoundError:
    print(f"❌ File not found: {input_csv}")
    print("   Make sure you've run the previous ABSA notebook first.")
    raise

# Sanity check: your Generational Comparison charts need this column to survive
if 'generation' not in df.columns:
    print("⚠️  WARNING: no 'generation' column found in this file — double check the notebook that produced it.")

df.head()

🔄 Loading data from: ../data/comment_and_aspects/aspect_detection_results.csv
✅ Successfully loaded 28936 aspect-tagged comments.
📋 Available columns: ['comment_id', 'generation', 'original_comment', 'detected_aspects', 'aspect_count', 'matched_keywords']


,comment_id,generation,original_comment,detected_aspects,aspect_count,matched_keywords
0,0,A52,why galaxy a with helio g processor is awesome...,Performance,1,"snapdragon, snap, lag, gaming, performa"
1,1,A52,gw dapet di harga rb hasil tt sama infinix hot...,"Features, Price, Storage, Connectivity",4,"nfc, harga, ram"
2,4,A52,secen di harga kemahalan ngak ya,Price,1,"harga, mahal, kemahalan"
3,7,A52,dpt harga brp bg,Price,1,harga
4,10,A52,alhamdulillah bisa nonton video ini pakai gala...,Camera,1,video


### 2. Load IndoBERT Sentiment Model

In [2]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

print("🤖 Loading BERT sentiment model...")

try:
    # Model untuk sentiment analysis Bahasa Indonesia
    model_name = "mdhugol/indonesia-bert-sentiment-classification"

    # Load model dan tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    # Buat pipeline
    sentiment_analyzer = pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        device=0 if torch.cuda.is_available() else -1  # Gunakan GPU jika ada
    )

    print("✅ BERT model loaded successfully!")
    print(f"   Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n🔧 Fallback: Using simpler model...")

    # Fallback ke model yang lebih kecil
    model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
    sentiment_analyzer = pipeline("sentiment-analysis", model=model_name)
    print(f"✅ Loaded fallback model: {model_name}")

# Mapping label BERT (sesuai model indonesia-bert-sentiment-classification)
label_map = {
    'LABEL_0': 'positive',
    'LABEL_1': 'neutral',
    'LABEL_2': 'negative'
}

/home/linuxsmol/projects/pythons/productsentiment/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖 Loading BERT sentiment model...


Loading weights: 100%|██████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 9905.83it/s]


✅ BERT model loaded successfully!
   Device: GPU


### 3. Run IndoBERT Sentiment Inference

In [3]:
from tqdm import tqdm

print("\n🤖 Starting IndoBERT Sentiment Inference...")

sentiments = []
scores = []

# Loop through the original comments to predict sentiment
for text in tqdm(df['original_comment'].astype(str).tolist(), desc="Predicting Sentiments"):
    try:
        result = sentiment_analyzer(text[:512])[0]  # Truncate as a safety margin under the model's 512-token limit

        # Convert LABEL_0/1/2 into readable positive/neutral/negative
        sentiments.append(label_map.get(result['label'], result['label']))
        scores.append(result['score'])
    except Exception as e:
        sentiments.append("neutral")  # Fallback for any weird errors
        scores.append(0.0)

# Add the predictions back into the dataframe
df['sentiment'] = sentiments
df['confidence_score'] = scores

print(f"✅ Done — predicted sentiment for {len(df)} comments.")
df[['original_comment', 'sentiment', 'confidence_score']].head()


🤖 Starting IndoBERT Sentiment Inference...


Predicting Sentiments: 100%|█████████████████████████████████████████████████████| 28936/28936 [03:12<00:00, 150.69it/s]

✅ Done — predicted sentiment for 28936 comments.


,original_comment,sentiment,confidence_score
0,why galaxy a with helio g processor is awesome...,neutral,0.830156
1,gw dapet di harga rb hasil tt sama infinix hot...,positive,0.839060
2,secen di harga kemahalan ngak ya,negative,0.992806
3,dpt harga brp bg,neutral,0.884975
4,alhamdulillah bisa nonton video ini pakai gala...,neutral,0.941752


### 4. Save Final Generational Sentiment Dataset

In [4]:
# ==========================================
# 3. SAVE THE FINAL GENERATIONAL DATASET
# ==========================================
os.makedirs("../data/final_results", exist_ok=True)

output_csv = "../data/final_results/final_generational_sentiment.csv"
df.to_csv(output_csv, index=False)

print(f"\n📁 FINAL DATASET SAVED TO: {output_csv}")
print("📊 Final Data Preview:")

preview_cols = [c for c in ['generation', 'detected_aspects', 'original_comment', 'sentiment'] if c in df.columns]
display(df[preview_cols].head())


📁 FINAL DATASET SAVED TO: ../data/final_results/final_generational_sentiment.csv
📊 Final Data Preview:


,generation,detected_aspects,original_comment,sentiment
0,A52,Performance,why galaxy a with helio g processor is awesome...,neutral
1,A52,"Features, Price, Storage, Connectivity",gw dapet di harga rb hasil tt sama infinix hot...,positive
2,A52,Price,secen di harga kemahalan ngak ya,negative
3,A52,Price,dpt harga brp bg,neutral
4,A52,Camera,alhamdulillah bisa nonton video ini pakai gala...,neutral
